In [ ]:
import json, voyageai, pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import paired_cosine_distances
from dotenv import load_dotenv

# VoyageAI API Key to run cosine similarity cells
load_dotenv()
vo = voyageai.Client()

In [2]:
extracted_json_files = pd.read_json('/Users/alexanderperalta/Desktop/Spring 2026/Projects/pa-demo/doc_intel/data.json')

for i in range(len(extracted_json_files['extracted_fields'].keys())):
    extracted_json_files['extracted_fields'][i]['source_file'] = extracted_json_files['metadata'][i]['source_file']

extracted_data = pd.DataFrame()

for i in range(len(extracted_json_files['extracted_fields'].keys())):
    extracted_data = pd.concat([extracted_data, pd.DataFrame(extracted_json_files['extracted_fields'][i], index=[0])])

extracted_data.sort_values(by='source_file', inplace=True)

In [3]:
manifest = Path('/Users/alexanderperalta/Desktop/Spring 2026/Projects/pa-demo/sample_docs/cases/manifest.json')

expected_data = pd.DataFrame()

with open(manifest, 'r', encoding='utf-8') as file:
    data = json.load(file)

for case in range(len(data['cases'])):
    data['cases'][case]['expected']['source_file'] = data['cases'][case]['case_id']
    expected_data = pd.concat([expected_data, pd.DataFrame(data['cases'][case]['expected'], index=[0])])

expected_data.sort_values(by='source_file', inplace=True)

In [4]:
extracted_data = extracted_data.replace({None: 0})
expected_data = expected_data.replace({None: 0})

In [5]:
embedding_dfs = [expected_data, extracted_data]
embedding_columns = ['medical_necessity_summary', 'primary_treatment_goal', 'diagnosis_description']

for df in embedding_dfs:
    for column in embedding_columns:
        response = vo.embed(list(df[column]), model="voyage-4-large", input_type="document")
        df[f'{column}_embedding'] = response.embeddings

In [7]:
summary_columns = ['medical_necessity_summary_embedding', 
                   'primary_treatment_goal_embedding', 
                   'diagnosis_description_embedding']
cosine_similarity_dict = {}

for column in embedding_columns:
    exp_matrix = expected_data[f'{column}_embedding'].tolist()
    ext_matrix = extracted_data[f'{column}_embedding'].tolist()
    
    doc_scores = 1 - paired_cosine_distances(exp_matrix, ext_matrix)
    extracted_data[f'{column}_semantic_score'] = doc_scores
    
    true_mean = doc_scores.mean()
    cosine_similarity_dict[column] = true_mean

cosine_similarity_df = pd.DataFrame(cosine_similarity_dict, index=[0]).transpose().rename(columns={0: 'cosine_similarity'})

In [8]:
cosine_similarity_df

,cosine_similarity
medical_necessity_summary,0.934382
primary_treatment_goal,0.715097
diagnosis_description,0.942227


In [13]:
metrics_report = {}
metrics_report_ignore = {'medical_necessity_summary', 
                         'primary_treatment_goal', 
                         'diagnosis_description', 
                         'medical_necessity_summary_embedding', 
                         'primary_treatment_goal_embedding',
                         'diagnosis_description_embedding',
                         'source_file'}
    
for column in expected_data.drop(columns=metrics_report_ignore).columns:
    if column not in extracted_data.columns:
        continue
        
    # Force strings to handle None/NaN safely
    y_true = expected_data[column].astype(str)
    y_pred = extracted_data[column].astype(str)
    
    match_rate = (y_true == y_pred).mean()

    metrics_report[column] = {'Match Rate': f'{match_rate * 100:.1f}%'}

metrics_df = pd.DataFrame(metrics_report).T
metrics_df

,Match Rate
patient_name,100.0%
dob,100.0%
diagnosis_code,100.0%
cpt_code,100.0%
provider_name,100.0%
provider_npi,100.0%
payer,100.0%
auth_period,100.0%
